# Video decomposition test
In this workbook you can decompose video file using the previously generated Gabor library space, similar to Skriabine S. et al 2026. Then visualize results.

In [28]:
from pathlib import Path
import numpy as np
from analysis_utils import downscale_binary_video
from wavelet_utils_vSpeed import loadFilterParamDict_vS, compute_and_save_dwt_vS, makeFilterParamDict_vS, saveFilterParamDict_vS, filename_fromFilterParam, get_filter_from_params

In [29]:
full_screen_coverage = [0, 88, -17, 49] # [az_left, az_right, el_bottom, el_top] full screen position in visual degrees
visual_coverage = [0, 88, -17, 49] # [az_left, az_right, el_bottom, el_top] screen coverage in visual degrees (seen by the animal)

screen_x = 100 # horizontal screen size in pixels for the Gabor filter generation and movie analysis

nx = 16 # number of Gabor filters in the horizontal direction (azimuth) (y will be generated)
        # 40 is a good number. Use 5 for testing to make it fast
        
n_thetas = 8 # number of angles to generate
theta_max = np.pi

size_min = 5 # minimum size in visual degrees
size_max = 25 # maximum size in visual degrees
n_sizes = 5   # number of sizes to generate

freq_min = .025 # minimum frequency in cycles per visual degree
freq_max = .15 # maximum frequency in cycles per visual degree
n_freqs = 5  # number of frequencies to generate

n_phases = 2  # number of phases to generate
phase_max = np.pi*1 # maximum phase to generate


In [30]:
#calculations
az_left, az_right, el_bottom, el_top = visual_coverage

screen_y = int(screen_x * (el_top - el_bottom) / (az_right - az_left))
ny = int(nx * (el_top - el_bottom) / (az_right - az_left))

# centers in visual degrees
xs = np.linspace(az_left, az_right, nx, endpoint=False)+(az_right - az_left) / (2*nx)
ys = np.linspace(el_bottom, el_top, ny, endpoint=False)+(el_top - el_bottom) / (2*ny)

angles= np.linspace(0, theta_max, n_thetas, endpoint=False)
sizes = np.logspace(np.log10(size_min), np.log10(size_max), n_sizes)
freqs = np.logspace(np.log10(freq_min), np.log10(freq_max), n_freqs)
phases = np.linspace(0,  phase_max, n_phases, endpoint=False)

print(f"Screen size: {screen_x}x{screen_y} pixels")
print(f"Full screen coverage: {full_screen_coverage} degrees")
print(f"Visual coverage: {visual_coverage} degrees")
print(f"Center positions (x_deg): {np.round(xs, 1)} degrees")
print(f"Center positions (y_deg): {np.round(ys, 1)} degrees")
print(f"Angles (degrees): {np.round(np.rad2deg(angles), 1)}")
print(f"Sizes (degrees): {sizes}")
print(f"Frequencies (cycles/degree): {freqs}")
print(f"Phases (degrees): {np.rad2deg(phases)}")


Screen size: 100x75 pixels
Full screen coverage: [0, 88, -17, 49] degrees
Visual coverage: [0, 88, -17, 49] degrees
Center positions (x_deg): [ 2.8  8.2 13.8 19.2 24.8 30.2 35.8 41.2 46.8 52.2 57.8 63.2 68.8 74.2
 79.8 85.2] degrees
Center positions (y_deg): [-14.2  -8.8  -3.2   2.2   7.8  13.2  18.8  24.2  29.8  35.2  40.8  46.2] degrees
Angles (degrees): [  0.   22.5  45.   67.5  90.  112.5 135.  157.5]
Sizes (degrees): [ 5.          7.47674391 11.18033989 16.71850762 25.        ]
Frequencies (cycles/degree): [0.025      0.03912711 0.06123724 0.09584147 0.15      ]
Phases (degrees): [ 0. 90.]


In [31]:
total_n=len(sizes)*len(angles)*len(freqs)*len(phases)*len(xs)*len(ys)
print(f"Total number of Gabor filters to generate: {total_n}")

Total number of Gabor filters to generate: 76800


In [32]:
# a control
gabor_step=(az_right-az_left)/nx 
print(f"Control: Gabor placement step in visual degrees (x): {gabor_step:.1f}, vs size_min: {size_min:.1f} degrees. {'OK' if (gabor_step < size_min) else 'WARNING!'}")
gabor_step=(el_top-el_bottom)/ny
print(f"Control: Gabor placement step in visual degrees (y): {gabor_step:.1f}, vs size_min: {size_min:.1f} degrees. {'OK' if (gabor_step < size_min) else 'WARNING!'}")
visual_step_x=(az_right-az_left)/screen_x
print(f"Control: Gabor resolution in visual degrees (x): {visual_step_x:.1f}, vs 1/freq_max: {1/freq_max:.1f} degrees. {'OK' if (visual_step_x < 1/freq_max/4) else 'WARNING!'}")
visual_step_y=(el_top-el_bottom)/screen_y
print(f"Control: Gabor resolution in visual degrees (y): {visual_step_y:.1f}, vs 1/freq_max: {1/freq_max:.1f} degrees. {'OK' if (visual_step_y < 1/freq_max/4) else 'WARNING!'}")


Control: Gabor placement step in visual degrees (x): 5.5, vs size_min: 5.0 degrees. WARNING!
Control: Gabor placement step in visual degrees (y): 5.5, vs size_min: 5.0 degrees. WARNING!
Control: Gabor resolution in visual degrees (x): 0.9, vs 1/freq_max: 6.7 degrees. OK
Control: Gabor resolution in visual degrees (y): 0.9, vs 1/freq_max: 6.7 degrees. OK


In [33]:

temppath = r'D:\SynologyDriveSyncedDATA\PROCESSED\Waven'

wavelet_params=makeFilterParamDict_vS(screen_x, screen_y, visual_coverage, full_screen_coverage, xs, ys, angles, sizes, freqs, phases)

_, paramname = filename_fromFilterParam(wavelet_params)
paramspath = Path(temppath) / paramname
saveFilterParamDict_vS(wavelet_params, paramspath)
print(f"Saved Gabor filter parameters to {paramspath}")


videopath = Path(temppath) / 'zebra_s0_d420.0_fps59.94_RESAMPLED30fps.mp4'
#videopath = Path(temppath) / 'zebra_s1_d420.0_fps59.94_RESAMPLED30fps.mp4'

Saved Gabor filter parameters to D:\SynologyDriveSyncedDATA\PROCESSED\Waven\gaborLibrary_vS_16_12_8_5_5_2.99dc00f0.json


In [34]:

xs, ys, angles, sizes, freqs,  phases, visual_coverage, full_screen_coverage,  screen_x, screen_y = loadFilterParamDict_vS(paramspath)


In [35]:
print(f"full_screen_coverage: {full_screen_coverage}, visual_coverage: {visual_coverage}")

full_screen_coverage: [  0  88 -17  49], visual_coverage: [  0  88 -17  49]


## Downsample video

In [36]:
downsampled_video_path=downscale_binary_video(videopath, full_screen_coverage, visual_coverage, screen_x, screen_y, force=False, fps=30)

Generating cropped and downsampled binary video...
Output file D:\SynologyDriveSyncedDATA\PROCESSED\Waven\zebra_s0_d420.0_fps59.94_RESAMPLED30fps_scaled100x75.npy already exists. Skipping generation.


## Display downsampled video result

In [37]:
# Displaying original and downsampled video frames
import cv2
import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import interact, IntSlider
from pathlib import Path

input_video_path = Path(videopath)
#output_npy_path 

out_video = np.load(downsampled_video_path, mmap_mode="r")

cap = cv2.VideoCapture(str(input_video_path))
n_frames = min(int(cap.get(cv2.CAP_PROP_FRAME_COUNT)), out_video.shape[0])

def show_frame(frame_idx):
    cap.set(cv2.CAP_PROP_POS_FRAMES, frame_idx)
    ret, frame = cap.read()
    if not ret:
        print(f"Could not read frame {frame_idx}")
        return

    input_gray = frame[:, :, 0]
    output_frame = out_video[frame_idx].T

    fig, axes = plt.subplots(1, 2, figsize=(10, 4))

    axes[0].imshow(
        input_gray,
        cmap="gray",
        extent=[full_screen_coverage[0], full_screen_coverage[1],
                full_screen_coverage[2], full_screen_coverage[3]],
        aspect="equal",
    )
    axes[0].set_title(f"Input frame {frame_idx}")
    axes[0].set_xlabel("Azimuth (°)")
    axes[0].set_ylabel("Elevation (°)")

    axes[1].imshow(
        output_frame,
        cmap="gray",
        vmin=0,
        vmax=1,
        extent=[visual_coverage[0], visual_coverage[1],
                visual_coverage[2], visual_coverage[3]],
        aspect="equal",
        origin="lower"
    )
    axes[1].set_title(f"Output frame {frame_idx}")
    axes[1].set_xlabel("Azimuth (°)")
    axes[1].set_ylabel("Elevation (°)")

    plt.tight_layout()
    plt.show()

interact(
    show_frame,
    frame_idx=IntSlider(
        value=0,
        min=0,
        max=n_frames - 1,
        step=1,
        description="Frame"
    )
)

interactive(children=(IntSlider(value=0, description='Frame', max=12599), Output()), _dom_classes=('widget-int…

<function __main__.show_frame(frame_idx)>

## Decomposition

In [38]:
if 'WT' in globals():
    del WT # unlink the memmapped variable as we might want to regenerate the file
    
dwt_path=compute_and_save_dwt_vS(downsampled_video_path, wavelet_params,   device='cuda', force=False)

Loaded downsampled video data from D:\SynologyDriveSyncedDATA\PROCESSED\Waven\zebra_s0_d420.0_fps59.94_RESAMPLED30fps_scaled100x75.npy with shape (12600, 100, 75) and dtype bool
||Constructing: zebra_s0_d420.0_fps59.94_RESAMPLED30fps_scaled100x75_libvS_16_12_8_5_5_2.99dc00f0dwt.npy 
Wavelet transform file D:\SynologyDriveSyncedDATA\PROCESSED\Waven\zebra_s0_d420.0_fps59.94_RESAMPLED30fps_scaled100x75_libvS_16_12_8_5_5_2.99dc00f0dwt.npy already exists. Skipping computation.


## Load and display decomposition

In [39]:
WT=np.load(dwt_path, mmap_mode='r')
print(f"Loaded DWT from {dwt_path} with shape {WT.shape}")

downsampled_video=np.load(downsampled_video_path)

Loaded DWT from D:\SynologyDriveSyncedDATA\PROCESSED\Waven\zebra_s0_d420.0_fps59.94_RESAMPLED30fps_scaled100x75_libvS_16_12_8_5_5_2.99dc00f0dwt.npy with shape (12600, 16, 12, 8, 5, 5, 2)


In [40]:
# interactive visualization of the DWT and video frames
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display

az_left, az_right, el_bottom, el_top = visual_coverage

param_names = ["x", "y", "angle", "size", "freq", "phase"]
param_values = [xs, ys, angles, sizes, freqs,  phases]

if WT.shape[0] != downsampled_video.shape[0]:
    print(f"Warning: WT has {WT.shape[0]} frames but video has {downsampled_video.shape[0]} frames!")
n_frames = downsampled_video.shape[0]

sliders = [
    widgets.IntSlider(
        value=len(vals) // 2,
        min=0,
        max=len(vals) - 1,
        step=1,
        description=name,
        continuous_update=False,
        layout=widgets.Layout(width="250px")
    )
    for name, vals in zip(param_names, param_values)
]

time_slider = widgets.IntSlider(
    value=0,
    min=0,
    max=n_frames - 1,
    step=1,
    description="frame",
    continuous_update=False,
    layout=widgets.Layout(width="250px")
)

out_zoom = widgets.Output()
out_full = widgets.Output()
out_overlay = widgets.Output()
out_xy = widgets.Output()

def plot_view(*args):
    xi, yi, anglei, sizei, freqi,  phasei = [s.value for s in sliders]
    ti = time_slider.value

    frame = downsampled_video[ti]
    filt = get_filter_from_params(xi, yi, anglei, sizei, freqi,  phasei, wavelet_params)


    transient = WT[:, xi, yi, anglei, sizei, freqi,   phasei]

    # x-y plane at current time and selected non-spatial parameters
    xy_plane = WT[ti, :, :, anglei, sizei, freqi,   phasei]

    title_params = (
        f"frame={ti}, "
        f"x={xs[xi]:.2f}, y={ys[yi]:.2f}, "
        f"angle={angles[anglei]:.2f}, "
        f"size={sizes[sizei]:.2f}, "
        f"freq={freqs[freqi]:.3f}, "
        f"phase={phases[phasei]:.2f}"
    )

    # --- TOP RIGHT: temporal zoom plot
    with out_zoom:
        out_zoom.clear_output(wait=True)

        z0 = max(0, ti - 12)
        z1 = min(n_frames, ti + 13)

        fig, ax = plt.subplots(1,1,figsize=(6, 3))
        ax.plot(np.arange(z0, z1), transient[z0:z1], marker="o")
        ax.axvline(ti, color="red", linestyle="--", linewidth=2)
        ax.set_xlim(z0, z1 - 1)
        ax.set_title("WT transient — zoom ±12 frames")
        ax.set_xlabel("Frame")
        ax.set_ylabel("Response")
        ax.set_ylim(transient.min() , transient.max() )
        plt.tight_layout()
        plt.show()

    # --- MIDDLE: full temporal plot
    with out_full:
        out_full.clear_output(wait=True)

        fig, ax = plt.subplots(figsize=(12, 3))
        ax.plot(transient)
        ax.axvline(ti, color="red", linestyle="--", linewidth=2)
        ax.set_xlim(0, n_frames - 1)
        ax.set_title("WT transient — full")
        ax.set_xlabel("Frame")
        ax.set_ylabel("Response")
        plt.tight_layout()
        plt.show()

    # --- BOTTOM LEFT: current video frame with Gabor overlay
    with out_overlay:
        out_overlay.clear_output(wait=True)

        fig, ax = plt.subplots(figsize=(6, 4))

        ax.imshow(
            frame.T,
            cmap="gray",
            extent=[az_left, az_right, el_bottom, el_top],
            origin="lower",
            aspect="auto",
        )

        filt_plot = filt.T
        v = np.max(np.abs(filt_plot))
        if v == 0:
            v = 1

        rgba = np.zeros((*filt_plot.shape, 4), dtype=float)
        pos = filt_plot > 0
        neg = filt_plot < 0

        rgba[pos, 0] = 1.0   # red = positive
        rgba[neg, 1] = 1.0   # green = negative
        rgba[..., 3] = np.abs(filt_plot) / v * 0.7

        ax.imshow(
            rgba,
            extent=[az_left, az_right, el_bottom, el_top],
            origin="lower",
            aspect="auto",
        )
        

        ax.set_title("Video + selected Gabor overlay")
        ax.set_xlabel("Azimuth (°)")
        ax.set_ylabel("Elevation (°)")
        plt.tight_layout()
        plt.show()

    # --- BOTTOM RIGHT: x-y WT plane
    with out_xy:
        out_xy.clear_output(wait=True)

        fig, ax = plt.subplots(figsize=(6, 4))

        im = ax.imshow(
            xy_plane.T,
            cmap="seismic",
            vmin=-0.4,
            vmax=0.4,
            extent=[az_left, az_right, el_bottom, el_top],
            origin="lower",
            aspect="auto",
        )


        ax.scatter(xs[xi], ys[yi], c="yellow", s=60, edgecolors="black")
        ax.set_title("WT x-y plane at selected frame")
        ax.set_xlabel("Azimuth (°)")
        ax.set_ylabel("Elevation (°)")
        plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
        plt.tight_layout()
        plt.show()

    sliders[0].description = "x"
    sliders[1].description = "y"

# update on slider movement
for s in sliders + [time_slider]:
    s.observe(plot_view, names="value")

slider_box = widgets.VBox([time_slider] + sliders)

top_row = widgets.HBox([slider_box, out_zoom])
bottom_row = widgets.HBox([out_overlay, out_xy])

display(top_row, out_full, bottom_row)

plot_view()

Output()